In [ ]:
!pip install huggingface_hub peft bitsandbytes transformers -q
!git clone https://github.com/flaviodell/llm-cv-finetuning-pipeline.git
%cd llm-cv-finetuning-pipeline
import sys
sys.path.insert(0, '/kaggle/working/llm-cv-finetuning-pipeline')

In [ ]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("HF Token:")

In [ ]:
import os
os.makedirs("models/llm/best_lora_adapter", exist_ok=True)
os.makedirs("outputs/llm", exist_ok=True)
os.makedirs("outputs/cv", exist_ok=True)
print("Paths created.")

In [ ]:
import os
print(os.listdir("models/llm/best_lora_adapter"))

In [ ]:
import json
from pathlib import Path
from huggingface_hub import HfApi, create_repo

token   = os.environ["HF_TOKEN"]
api     = HfApi()
hf_user = api.whoami(token=token)["name"]
repo_id = f"{hf_user}/pet-expert-mistral7b-lora"

print(f"Pushing LLM adapter to: https://huggingface.co/{repo_id}")

create_repo(
    repo_id=repo_id,
    token=token,
    private=False,
    exist_ok=True
)

# Load LLM evaluation results dynamically
with open("outputs/llm/llm_results.json", encoding="utf-8") as f:
    llm_results = json.load(f)

model_card = f"""---
language: en
license: mit
tags:
  - text-generation
  - pytorch
  - mistral
  - lora
  - peft
  - veterinary
  - fine-tuning
base_model: mistralai/Mistral-7B-Instruct-v0.3
---

# Pet Expert — Mistral 7B + LoRA

Fine-tuned Mistral 7B Instruct on a synthetic veterinary dataset
covering 37 cat and dog breeds from the Oxford-IIIT Pet dataset.

## Model details

- **Base model**: mistralai/Mistral-7B-Instruct-v0.3
- **Method**: LoRA (r=16, alpha=32) with 4-bit NF4 quantization
- **Dataset**: Synthetic — 185 instruction/response pairs
- **Domain**: Veterinary — breed temperament, health, training advice
- **Framework**: HuggingFace PEFT + bitsandbytes

## Results (RAGAS evaluation)

| Metric | Value |
|--------|-------|
| Faithfulness | {llm_results.get('faithfulness', 'N/A')} |
| Answer relevancy | {llm_results.get('answer_relevancy', 'N/A')} |
| Evaluated on | {llm_results.get('num_questions', 'N/A')} questions |

## Related model

This LLM adapter is part of a two-model project.
The companion CV classifier is available at:
`{hf_user}/oxford-pets-resnet50`

## Usage

```python
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained("{repo_id}")
base_model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.3",
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "{repo_id}")
model.eval()

prompt = (
    "<|system|>\\nYou are an expert veterinarian.\\n"
    "<|user|>\\nWhat are the health concerns for a Persian cat?\\n"
    "<|assistant|>\\n"
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=300)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))
```
"""

model_card_path = Path("outputs/llm/MODEL_CARD.md")
with open(model_card_path, "w", encoding="utf-8") as f:
    f.write(model_card)

# Upload adapter files
adapter_dir = Path("models/llm/best_lora_adapter")
for file_path in adapter_dir.iterdir():
    if file_path.is_file():
        print(f"Uploading {file_path.name}...")
        api.upload_file(
            path_or_fileobj=str(file_path),
            path_in_repo=file_path.name,
            repo_id=repo_id,
            token=token
        )

# Upload model card
api.upload_file(
    path_or_fileobj=str(model_card_path),
    path_in_repo="README.md",
    repo_id=repo_id,
    token=token
)

print(f"\nLLM adapter pushed successfully.")
print(f"View at: https://huggingface.co/{repo_id}")